# Fine-tune Qwen2.5-1.5B-Instruct (LoRA) trên UIT-ViQuAD2.0

Notebook end-to-end cho **Extractive QA** (SQuAD 2.0 style) trên [taidng/UIT-ViQuAD2.0](https://huggingface.co/datasets/taidng/UIT-ViQuAD2.0).

**Pipeline:** tải HF dataset → profiling token length → LoRA (Unsloth, **không quantize**) → lưu adapter → đánh giá EM/F1.

**Hardware mục tiêu:** RTX 5060 Ti 15GB VRAM (full-precision LoRA fp16/bf16, `max_seq_length` tự động từ P99).

**Câu không có đáp án** (`is_impossible=true`): target = `"Không có đáp án trong đoạn văn"`.

### Chạy trên máy thuê (RTX 5060 Ti 15GB)
1. Mở notebook tại `UIT-ViQuAD2.0/qwen2.5-1.5b/LoRa/`
2. Chạy **Run All** (cần GPU + internet lần đầu)
3. Nếu **CUDA OOM**: trong cell train đổi `per_device_train_batch_size=1`, `gradient_accumulation_steps=8`; hoặc hạ `MAX_SEQ_CAP=3584`
4. Chỉ evaluate adapter đã lưu: `RUN_TRAINING=False`, `RUN_EVALUATION=True`

In [ ]:
# Bước 1: Gỡ cài đặt các gói cũ để tránh xung đột phiên bản
!pip uninstall torch torchvision torchaudio xformers transformers trl unsloth unsloth_zoo -y

In [ ]:
# Bước 2: Cài PyTorch (CUDA 12.8 — tương thích CUDA 12.x trên máy thuê)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128 --no-cache-dir

In [ ]:
# Bước 3: Cài transformers, trl, peft, accelerate, bitsandbytes, xformers, unsloth
!pip install transformers trl peft accelerate bitsandbytes xformers datasets --no-cache-dir
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --no-cache-dir
!pip install unsloth_zoo scikit-learn --no-cache-dir

In [ ]:
import json
import math
import os
import string
import unicodedata
from collections import Counter
from pathlib import Path

import torch
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoTokenizer

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM: {props.total_memory / 1024**3:.1f} GB")

# ==========================================
# CẤU HÌNH CHUNG
# ==========================================
BASE_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
DATASET_NAME = "taidng/UIT-ViQuAD2.0"
NO_ANSWER_SENTINEL = "Không có đáp án trong đoạn văn"

SYSTEM_PROMPT = (
    "Bạn là một trợ lý AI thông minh, giỏi tiếng Việt. "
    "Nhiệm vụ của bạn là đọc đoạn văn và trả lời câu hỏi "
    "bằng cách trích xuất chính xác đoạn văn bản ngắn nhất từ ngữ cảnh được cung cấp. "
    "Nếu câu hỏi không thể trả lời từ đoạn văn, hãy trả lời: "
    f"{NO_ANSWER_SENTINEL}."
)

USER_PROMPT_TEMPLATE = (
    "Đọc đoạn văn sau và trả lời câu hỏi bằng cách trích xuất "
    "đúng cụm từ xuất hiện trong đoạn văn.\n\n"
    "Đoạn văn:\n{context}\n\n"
    "Câu hỏi: {question}\n\n"
    "Hãy trả lời ngắn gọn, chỉ là cụm từ trích xuất từ đoạn văn "
    f"hoặc '{NO_ANSWER_SENTINEL}' nếu không có đáp án:"
)

LORA_SAVE_PATH = "qwen2.5-1.5b-instruct-lora-viquad2"
EVAL_RESULTS_PATH = "eval_results_viquad2.json"
PROFILING_CONFIG_PATH = "profiling_config.json"

# Đường dẫn export JSON tại root dataset (../../ từ thư mục LoRa)
DATASET_ROOT = Path("../../")
DEV_JSON_PATH = DATASET_ROOT / "dev_viquad2.json"
TEST_JSON_PATH = DATASET_ROOT / "test_viquad2.json"

# Huấn luyện: đặt False nếu chỉ chạy evaluation với adapter đã lưu
RUN_TRAINING = True
RUN_EVALUATION = True

# LoRA thường (không QLoRA)
LOAD_IN_4BIT = False
LOAD_IN_8BIT = False

# VRAM cap cho max_seq_length (15GB fp16 LoRA; hạ MAX_SEQ_CAP hoặc batch nếu OOM)
MAX_SEQ_CAP = 4096
MIN_SEQ_LENGTH = 512

In [ ]:
# ==========================================
# TẢI DATASET TỪ HUGGINGFACE & EXPORT JSON
# ==========================================

def row_to_sample(row):
    """Chuyển 1 dòng HF thành dict chuẩn hóa."""
    is_impossible = bool(row["is_impossible"])
    if is_impossible:
        answer = NO_ANSWER_SENTINEL
    else:
        texts = row["answers"]["text"] if row["answers"] else []
        answer = texts[0] if texts else NO_ANSWER_SENTINEL
        if not answer.strip():
            answer = NO_ANSWER_SENTINEL
            is_impossible = True
    return {
        "id": row.get("id", ""),
        "context": row["context"],
        "question": row["question"],
        "answer": answer,
        "is_impossible": is_impossible,
    }


def hf_split_to_samples(split_dataset):
    return [row_to_sample(row) for row in split_dataset]


def save_samples_json(samples, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(samples, f, ensure_ascii=False, indent=2)
    print(f"Đã lưu {len(samples)} mẫu → {path.resolve()}")


try:
    print(f"Đang tải {DATASET_NAME} ...")
    raw_ds = load_dataset(DATASET_NAME)
    train_samples = hf_split_to_samples(raw_ds["train"])
    dev_samples = hf_split_to_samples(raw_ds["validation"])
    test_samples = hf_split_to_samples(raw_ds["test"])
except Exception as e:
    print(f"Không tải được từ HF ({e}). Dùng file JSON local nếu có.")
    if not TEST_JSON_PATH.exists():
        raise
    with open(TEST_JSON_PATH, "r", encoding="utf-8") as f:
        test_samples = json.load(f)
    with open(DEV_JSON_PATH, "r", encoding="utf-8") as f:
        dev_samples = json.load(f)
    train_samples = dev_samples  # chỉ dùng cho profiling khi thiếu train
    print("⚠ Đang dùng dev làm proxy train cho profiling — nên tải HF đầy đủ khi train.")

print(f"Train      : {len(train_samples)}")
print(f"Validation : {len(dev_samples)}")
print(f"Test       : {len(test_samples)}")

n_impossible_train = sum(1 for s in train_samples if s["is_impossible"])
print(f"Unanswerable (train): {n_impossible_train} ({100*n_impossible_train/len(train_samples):.1f}%)")

save_samples_json(dev_samples, DEV_JSON_PATH)
save_samples_json(test_samples, TEST_JSON_PATH)

print("\nVí dụ answerable:")
ex = next(s for s in train_samples if not s["is_impossible"])
print(f"  Q: {ex['question'][:80]}...")
print(f"  A: {ex['answer']}")

print("\nVí dụ unanswerable:")
ex2 = next(s for s in train_samples if s["is_impossible"])
print(f"  Q: {ex2['question'][:80]}...")
print(f"  A: {ex2['answer']}")

In [ ]:
# ==========================================
# PROFILING TOKEN LENGTH → max_seq_length
# ==========================================
from unsloth.chat_templates import get_chat_template

tokenizer_prof = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
tokenizer_prof = get_chat_template(tokenizer_prof, chat_template="qwen-2.5")


def build_messages(context, question, answer=None, for_inference=False):
    user_content = USER_PROMPT_TEMPLATE.format(context=context, question=question)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]
    if answer is not None and not for_inference:
        messages.append({"role": "assistant", "content": answer})
    return messages


def sample_to_train_text(sample, tok):
    messages = build_messages(sample["context"], sample["question"], sample["answer"])
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)


def compute_max_seq_length(samples, tok, cap=MAX_SEQ_CAP, min_len=MIN_SEQ_LENGTH):
    lengths = []
    for s in tqdm(samples, desc="Token profiling"):
        text = sample_to_train_text(s, tok)
        lengths.append(len(tok.encode(text)))
    lengths.sort()
    n = len(lengths)
    stats = {
        "min": lengths[0],
        "p50": lengths[n // 2],
        "p95": lengths[int(n * 0.95)],
        "p99": lengths[int(n * 0.99)],
        "max": lengths[-1],
    }
    chosen = min(math.ceil(stats["p99"] * 1.05), cap)
    chosen = ((chosen + 255) // 256) * 256
    chosen = max(chosen, min_len)
    truncated = sum(1 for L in lengths if L > chosen)
    stats["chosen_max_seq_length"] = chosen
    stats["truncated_samples"] = truncated
    stats["truncated_pct"] = round(100 * truncated / n, 3)
    return chosen, stats, lengths


if Path(PROFILING_CONFIG_PATH).exists() and not RUN_TRAINING:
    with open(PROFILING_CONFIG_PATH, "r", encoding="utf-8") as f:
        prof_cfg = json.load(f)
    max_seq_length = prof_cfg["max_seq_length"]
    length_stats = prof_cfg["token_length_stats"]
    print(f"Đã load profiling từ {PROFILING_CONFIG_PATH}: max_seq_length={max_seq_length}")
else:
    max_seq_length, length_stats, _ = compute_max_seq_length(train_samples, tokenizer_prof)
    with open(PROFILING_CONFIG_PATH, "w", encoding="utf-8") as f:
        json.dump({"max_seq_length": max_seq_length, "token_length_stats": length_stats}, f, indent=2)
    print(f"Đã lưu profiling → {PROFILING_CONFIG_PATH}")

print("\n--- Token length stats (train, full ChatML) ---")
for k, v in length_stats.items():
    print(f"  {k}: {v}")
print(f"\n=> max_seq_length = {max_seq_length}")
if length_stats.get("truncated_pct", 0) > 1.0:
    print(f"⚠ {length_stats['truncated_pct']}% mẫu sẽ bị truncate. Cân nhắc tăng MAX_SEQ_CAP nếu VRAM cho phép.")

In [ ]:
# ==========================================
# LOAD MODEL + LORA (Unsloth) — chỉ khi train
# ==========================================
if RUN_TRAINING:
    from unsloth import FastLanguageModel

    dtype = None  # None = auto fp16 (T4/V100) hoặc bf16 (Ampere+)

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL_NAME,
        max_seq_length=max_seq_length,
        dtype=dtype,
        load_in_4bit=LOAD_IN_4BIT,
        load_in_8bit=LOAD_IN_8BIT,
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
        use_rslora=False,
        loftq_config=None,
    )

    tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")
    print("Model + LoRA loaded (no quantize). max_seq_length =", max_seq_length)
else:
    print("RUN_TRAINING=False — bỏ qua load model huấn luyện.")

In [ ]:
# ==========================================
# CHUẨN BỊ DATASET CHO SFTTrainer
# ==========================================
if RUN_TRAINING:
    from datasets import Dataset

    def formatting_prompts_func(examples):
        texts = []
        for context, question, answer in zip(
            examples["context"], examples["question"], examples["answer"]
        ):
            messages = build_messages(context, question, answer)
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False
            )
            texts.append(text)
        return {"text": texts}

    train_hf = Dataset.from_list(train_samples)
    dataset = train_hf.map(
        formatting_prompts_func,
        batched=True,
        remove_columns=train_hf.column_names,
        desc="Formatting train prompts",
    )

    print(f"Training dataset: {len(dataset)} samples")
    print("\n--- Preview (first 600 chars) ---")
    print(dataset[0]["text"][:600])
else:
    print("RUN_TRAINING=False — bỏ qua chuẩn bị dataset.")

In [ ]:
# ==========================================
# HUẤN LUYỆN (Fine-Tuning)
# RTX 5060 Ti 15GB: batch=2, grad_accum=4 (effective=8)
# Nếu OOM: đổi per_device_train_batch_size=1, gradient_accumulation_steps=8
# ==========================================
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
import sys

if RUN_TRAINING:
    trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=dataset,
        dataset_text_field="text",
        max_seq_length=max_seq_length,
        dataset_num_proc=1,
        packing=False,
        args=TrainingArguments(
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            warmup_steps=10,
            num_train_epochs=3,
            learning_rate=2e-4,
            fp16=not is_bfloat16_supported(),
            bf16=is_bfloat16_supported(),
            logging_steps=20,
            optim="adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="cosine",
            seed=3407,
            output_dir="outputs_viquad2",
            save_strategy="epoch",
            report_to="none",
        ),
    )

    real_config_cls = type(trainer.args)
    real_trainer_cls = type(trainer)
    sys.modules["trl.trainer.sft_config"] = sys.modules[real_config_cls.__module__]
    sys.modules["trl.trainer.sft_trainer"] = sys.modules[real_trainer_cls.__module__]
    sys.modules[real_config_cls.__module__].SFTConfig = real_config_cls
    sys.modules[real_trainer_cls.__module__].SFTTrainer = real_trainer_cls

    print("\nBắt đầu fine-tuning...")
    trainer_stats = trainer.train()
    print("Training hoàn tất.")
else:
    print("SKIP_TRAINING=True — bỏ qua huấn luyện.")

In [ ]:
# ==========================================
# INFERENCE NHANH (smoke test)
# ==========================================
if RUN_TRAINING:
    from unsloth import FastLanguageModel
    FastLanguageModel.for_inference(model)

    def run_inference(context, question, tok, mdl, max_new_tokens=64):
        messages = build_messages(context, question, answer=None, for_inference=True)
        prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tok(prompt, return_tensors="pt", truncation=True, max_length=max_seq_length).to(mdl.device)
        with torch.no_grad():
            outputs = mdl.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                use_cache=True,
                pad_token_id=tok.pad_token_id,
                eos_token_id=tok.eos_token_id,
            )
        input_len = inputs["input_ids"].shape[1]
        pred = tok.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
        return pred.split("\n")[0].strip()

    ans_sample = next(s for s in train_samples if not s["is_impossible"])
    noans_sample = next(s for s in train_samples if s["is_impossible"])

    print("--- Answerable ---")
    print("Q:", ans_sample["question"][:100])
    print("Truth:", ans_sample["answer"])
    print("Pred:", run_inference(ans_sample["context"], ans_sample["question"], tokenizer, model))

    print("\n--- Unanswerable ---")
    print("Q:", noans_sample["question"][:100])
    print("Truth:", noans_sample["answer"])
    print("Pred:", run_inference(noans_sample["context"], noans_sample["question"], tokenizer, model))

In [ ]:
# ==========================================
# LƯU LORA ADAPTER
# ==========================================
if RUN_TRAINING:
    model.save_pretrained(LORA_SAVE_PATH)
    tokenizer.save_pretrained(LORA_SAVE_PATH)
    print(f"Đã lưu LoRA adapter tại: {LORA_SAVE_PATH}")
    # model.save_pretrained_merged("qwen2.5-1.5b-instruct-viquad2-merged", tokenizer, save_method="merged_16bit")

## Đánh giá trên tập Test

- **HasAns EM / F1:** chỉ câu có đáp án (`is_impossible=false`)
- **NoAns accuracy:** câu không có đáp án — prediction khớp sentinel
- **Overall:** trung bình trên toàn bộ test set

Đặt `RUN_TRAINING=False` và chạy lại từ cell cấu hình nếu chỉ evaluate adapter đã lưu.

In [ ]:
# ==========================================
# METRICS & EVALUATION HELPERS
# ==========================================

def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFC", text or "")
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return " ".join(text.split())


def compute_exact_match(prediction: str, ground_truth: str) -> int:
    return int(normalize_text(prediction) == normalize_text(ground_truth))


def compute_f1_token(prediction: str, ground_truth: str) -> float:
    pred_tokens = normalize_text(prediction).split()
    true_tokens = normalize_text(ground_truth).split()
    if not pred_tokens and not true_tokens:
        return 1.0
    if not pred_tokens or not true_tokens:
        return 0.0
    common = Counter(pred_tokens) & Counter(true_tokens)
    num_common = sum(common.values())
    if num_common == 0:
        return 0.0
    precision = num_common / len(pred_tokens)
    recall = num_common / len(true_tokens)
    return (2 * precision * recall) / (precision + recall)


def is_no_answer_prediction(prediction: str) -> bool:
    return normalize_text(prediction) == normalize_text(NO_ANSWER_SENTINEL)


def format_prompt_inference(context, question, tok):
    messages = build_messages(context, question, answer=None, for_inference=True)
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


print("Metric helpers ready.")

In [ ]:
# ==========================================
# LOAD MODEL CHO EVAL (hoặc dùng model vừa train)
# ==========================================
from transformers import AutoModelForCausalLM
from peft import PeftModel

if RUN_EVALUATION:
    if RUN_TRAINING:
        from unsloth import FastLanguageModel
        model_eval = model
        tokenizer_eval = tokenizer
        FastLanguageModel.for_inference(model_eval)
        print("Dùng model vừa train cho evaluation.")
    else:
        print("Loading adapter từ disk...")
        adapter_tok_path = LORA_SAVE_PATH if Path(LORA_SAVE_PATH).exists() else BASE_MODEL_NAME
        tokenizer_eval = AutoTokenizer.from_pretrained(adapter_tok_path)
        if not Path(LORA_SAVE_PATH).exists():
            tokenizer_eval = get_chat_template(tokenizer_eval, chat_template="qwen-2.5")
        base_eval = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL_NAME,
            torch_dtype=torch.float16,
            device_map="auto",
        )
        model_eval = PeftModel.from_pretrained(base_eval, LORA_SAVE_PATH)
        model_eval.eval()
        print("Load adapter hoàn tất.")

In [ ]:
# ==========================================
# CHẠY EVALUATION TRÊN TEST SET
# ==========================================
if RUN_EVALUATION:
    has_ans_em, has_ans_f1 = [], []
    no_ans_correct = []
    all_predictions = []

    print(f"Evaluating {len(test_samples)} test samples...\n")

    for sample in tqdm(test_samples, desc="Test eval"):
        prompt = format_prompt_inference(sample["context"], sample["question"], tokenizer_eval)
        inputs = tokenizer_eval(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=max_seq_length,
        ).to(model_eval.device)

        with torch.no_grad():
            outputs = model_eval.generate(
                **inputs,
                max_new_tokens=64,
                do_sample=False,
                pad_token_id=tokenizer_eval.pad_token_id,
                eos_token_id=tokenizer_eval.eos_token_id,
            )

        input_len = inputs["input_ids"].shape[1]
        prediction = tokenizer_eval.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
        prediction = prediction.split("\n")[0].strip()

        ground_truth = sample["answer"]
        if sample["is_impossible"]:
            no_ans_ok = int(is_no_answer_prediction(prediction))
            no_ans_correct.append(no_ans_ok)
            em, f1 = no_ans_ok, float(no_ans_ok)
        else:
            em = compute_exact_match(prediction, ground_truth)
            f1 = compute_f1_token(prediction, ground_truth)
            has_ans_em.append(em)
            has_ans_f1.append(f1)

        all_predictions.append({
            "id": sample["id"],
            "question": sample["question"],
            "is_impossible": sample["is_impossible"],
            "ground_truth": ground_truth,
            "prediction": prediction,
            "em": em,
            "f1": f1,
        })

    # Aggregate metrics
    has_em = 100 * sum(has_ans_em) / len(has_ans_em) if has_ans_em else 0.0
    has_f1 = 100 * sum(has_ans_f1) / len(has_ans_f1) if has_ans_f1 else 0.0
    no_acc = 100 * sum(no_ans_correct) / len(no_ans_correct) if no_ans_correct else 0.0
    overall_em = 100 * sum(p["em"] for p in all_predictions) / len(all_predictions)
    overall_f1 = 100 * sum(p["f1"] for p in all_predictions) / len(all_predictions)

    print("\n" + "=" * 58)
    print("   KẾT QUẢ ĐÁNH GIÁ - UIT-ViQuAD2.0 Test")
    print("   Model: Qwen2.5-1.5B-Instruct + LoRA (Unsloth)")
    print("=" * 58)
    print(f"  Test samples          : {len(test_samples)}")
    print(f"  HasAns (n)            : {len(has_ans_em)}")
    print(f"  NoAns  (n)            : {len(no_ans_correct)}")
    print(f"  HasAns EM             : {has_em:.2f}%")
    print(f"  HasAns F1             : {has_f1:.2f}%")
    print(f"  NoAns Accuracy        : {no_acc:.2f}%")
    print(f"  Overall EM            : {overall_em:.2f}%")
    print(f"  Overall F1            : {overall_f1:.2f}%")
    print(f"  max_seq_length        : {max_seq_length}")
    print("=" * 58)

    eval_results = {
        "dataset": DATASET_NAME,
        "base_model": BASE_MODEL_NAME,
        "adapter": LORA_SAVE_PATH,
        "max_seq_length": max_seq_length,
        "token_length_stats": length_stats,
        "hasans_em": round(has_em, 4),
        "hasans_f1": round(has_f1, 4),
        "noans_accuracy": round(no_acc, 4),
        "overall_em": round(overall_em, 4),
        "overall_f1": round(overall_f1, 4),
        "total": len(all_predictions),
        "predictions": all_predictions,
    }

    with open(EVAL_RESULTS_PATH, "w", encoding="utf-8") as f:
        json.dump(eval_results, f, ensure_ascii=False, indent=2)
    print(f"\nKết quả chi tiết: {EVAL_RESULTS_PATH}")